# Energy Resource Shock Simulator - AI App Prototype

**Goal:** build a student-friendly prototype that lets a user place war, disaster, earthquake, strike, cyberattack, port closure, or other supply-chain disturbance events on a global map and forecast possible near-future impacts on energy and commodity resources such as oil, natural gas, gold, copper, silver, gasoline, and heating oil.

This notebook is designed for **Google Colab** and uses free or optional public data sources where possible. It includes:

1. **Global event map** with user-defined crisis events and supply-chain hubs.
2. **Machine learning forecasting** using historical commodity price features plus simulated historical disruptions for training/demo validation.
3. **Deep learning / neural-network crisis classification** plus clustering of event descriptions.
4. **Reinforcement learning** to recommend response actions under disruption pressure.
5. **App packaging pathway** for Streamlit web app, GitHub repo, iOS/PWA route, copyright/licensing, and promotion.

> Important educational limitation: this is a simulation and teaching prototype, not an investment, emergency-response, military, or operational decision system. Real deployment requires verified event data, ground-truth supply-chain data, bias testing, cybersecurity review, model monitoring, and domain-expert validation.

## Recommended free/public data sources

The prototype supports or explains these feeds:

- **World Bank Commodity Markets / Pink Sheet** for monthly commodity prices.
- **GDELT** for global news/event context and locations.
- **EIA Open Data API v2** for energy data. EIA requires a free API key for API calls.
- **Yahoo Finance via yfinance** for quick student demos of historical commodity futures prices. This is convenient but not official market infrastructure.

For a classroom project, start with the built-in demo/synthetic data, then replace the synthetic event labels with a curated historical event dataset.

In [ ]:
# =====================================================================
# 1. Install dependencies for Colab
# =====================================================================
# Colab often already has many packages. This cell installs missing ones.
# Re-run this notebook after installation if Colab asks you to restart.

%pip install -q numpy pandas scikit-learn matplotlib plotly folium ipywidgets yfinance requests joblib streamlit

In [ ]:
# =====================================================================
# 2. Imports and notebook setup
# =====================================================================
import json
import math
import os
import random
import warnings
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from IPython.display import display, HTML

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print('Notebook ready. Running in Colab:', IN_COLAB)

In [ ]:
# =====================================================================
# 3. Resource universe, supply-chain hubs, and helper functions
# =====================================================================

RESOURCE_TICKERS = {
    'WTI Crude Oil': 'CL=F',
    'Brent Crude Oil': 'BZ=F',
    'Natural Gas': 'NG=F',
    'Gold': 'GC=F',
    'Silver': 'SI=F',
    'Copper': 'HG=F',
    'Gasoline': 'RB=F',
    'Heating Oil': 'HO=F',
}

EVENT_TYPES = [
    'war', 'earthquake', 'hurricane', 'port_closure', 'pipeline_failure',
    'cyberattack', 'labor_strike', 'sanctions', 'mine_accident', 'drought',
    'shipping_chokepoint', 'pandemic'
]

# Simplified global supply-chain hubs. Expand this table for a real project.
SUPPLY_HUBS = pd.DataFrame([
    {'hub': 'Strait of Hormuz', 'lat': 26.57, 'lon': 56.25, 'resources': 'WTI Crude Oil,Brent Crude Oil,Natural Gas,Gasoline,Heating Oil', 'importance': 10},
    {'hub': 'Suez Canal', 'lat': 30.59, 'lon': 32.27, 'resources': 'WTI Crude Oil,Brent Crude Oil,Natural Gas,Gasoline,Heating Oil', 'importance': 8},
    {'hub': 'Panama Canal', 'lat': 9.08, 'lon': -79.68, 'resources': 'WTI Crude Oil,Brent Crude Oil,Natural Gas,Copper,Gold', 'importance': 7},
    {'hub': 'US Gulf Coast', 'lat': 29.76, 'lon': -95.37, 'resources': 'WTI Crude Oil,Natural Gas,Gasoline,Heating Oil', 'importance': 9},
    {'hub': 'North Sea', 'lat': 57.0, 'lon': 2.5, 'resources': 'Brent Crude Oil,Natural Gas', 'importance': 8},
    {'hub': 'West Siberia', 'lat': 61.0, 'lon': 75.0, 'resources': 'WTI Crude Oil,Brent Crude Oil,Natural Gas', 'importance': 8},
    {'hub': 'Western Australia Mining', 'lat': -23.7, 'lon': 121.0, 'resources': 'Gold,Silver,Copper', 'importance': 6},
    {'hub': 'South Africa Gold Belt', 'lat': -26.2, 'lon': 28.0, 'resources': 'Gold', 'importance': 7},
    {'hub': 'Chile Copper Belt', 'lat': -22.5, 'lon': -68.9, 'resources': 'Copper,Gold,Silver', 'importance': 9},
    {'hub': 'Indonesia LNG / Mining', 'lat': -2.5, 'lon': 118.0, 'resources': 'Natural Gas,Copper,Gold', 'importance': 7},
    {'hub': 'Australia LNG', 'lat': -20.0, 'lon': 116.0, 'resources': 'Natural Gas', 'importance': 7},
    {'hub': 'Norway Energy Hub', 'lat': 60.4, 'lon': 5.3, 'resources': 'Brent Crude Oil,Natural Gas', 'importance': 7},
])

EVENT_BASE_IMPACT = {
    'war': 1.00,
    'earthquake': 0.65,
    'hurricane': 0.70,
    'port_closure': 0.75,
    'pipeline_failure': 0.85,
    'cyberattack': 0.55,
    'labor_strike': 0.45,
    'sanctions': 0.90,
    'mine_accident': 0.60,
    'drought': 0.40,
    'shipping_chokepoint': 0.95,
    'pandemic': 0.60,
}

RESOURCE_SENSITIVITY = {
    'WTI Crude Oil': 1.00,
    'Brent Crude Oil': 1.05,
    'Natural Gas': 0.90,
    'Gold': 0.55,       # often also reacts as a safe-haven asset
    'Silver': 0.45,
    'Copper': 0.70,
    'Gasoline': 0.85,
    'Heating Oil': 0.85,
}

SAFE_HAVEN_EVENTS = {'war', 'sanctions', 'pandemic', 'cyberattack'}


def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km."""
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2.0) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlambda / 2.0) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


def nearest_hub_features(lat: float, lon: float, resource: str) -> Dict[str, float | str]:
    subset = SUPPLY_HUBS[SUPPLY_HUBS['resources'].str.contains(resource, regex=False)].copy()
    if subset.empty:
        subset = SUPPLY_HUBS.copy()
    distances = haversine_km(lat, lon, subset['lat'].values, subset['lon'].values)
    idx = int(np.argmin(distances))
    hub = subset.iloc[idx]
    return {
        'nearest_hub': hub['hub'],
        'distance_to_hub_km': float(distances[idx]),
        'hub_importance': float(hub['importance']),
    }


def distance_decay(distance_km: float, half_life_km: float = 1500.0) -> float:
    """Higher if the event is closer to a relevant hub."""
    return float(np.exp(-distance_km / half_life_km))

print('Resources:', list(RESOURCE_TICKERS.keys()))
display(SUPPLY_HUBS)

In [ ]:
# =====================================================================
# 4. Optional online data: market prices, World Bank Pink Sheet, GDELT
# =====================================================================

WORLD_BANK_PINK_SHEET_MONTHLY_XLSX = (
    'https://thedocs.worldbank.org/en/doc/74e8be41ceb20fa0da750cda2f6b9e4e-0050012026/'
    'related/CMO-Historical-Data-Monthly.xlsx'
)


def make_sample_market_data(start='2018-01-01', periods=1600) -> pd.DataFrame:
    """Fallback sample price data so the notebook always runs."""
    rng = np.random.default_rng(RANDOM_SEED)
    dates = pd.bdate_range(start=start, periods=periods)
    params = {
        'WTI Crude Oil': (70, 0.03, 0.32),
        'Brent Crude Oil': (75, 0.03, 0.30),
        'Natural Gas': (3.0, 0.02, 0.55),
        'Gold': (1800, 0.04, 0.18),
        'Silver': (23, 0.03, 0.30),
        'Copper': (4.0, 0.03, 0.28),
        'Gasoline': (2.4, 0.03, 0.35),
        'Heating Oil': (2.6, 0.03, 0.34),
    }
    frames = []
    for resource, (base, annual_drift, annual_vol) in params.items():
        daily_returns = rng.normal(annual_drift / 252, annual_vol / np.sqrt(252), len(dates))
        # Add a few artificial shock pulses to create richer demo dynamics.
        shock = np.zeros(len(dates))
        for center in rng.choice(np.arange(120, len(dates)-120), size=6, replace=False):
            width = rng.integers(8, 35)
            amplitude = rng.normal(0, annual_vol / 8)
            window = np.arange(len(dates))
            shock += amplitude * np.exp(-((window - center) ** 2) / (2 * width ** 2))
        price = base * np.exp(np.cumsum(daily_returns + shock / 252))
        frames.append(pd.DataFrame({'date': dates, 'resource': resource, 'close': price}))
    return pd.concat(frames, ignore_index=True)


def download_yfinance_market_data(start='2016-01-01', end=None) -> pd.DataFrame:
    """Download commodity futures prices via yfinance. Returns empty frame on failure."""
    try:
        import yfinance as yf
    except Exception as e:
        print('yfinance import failed:', e)
        return pd.DataFrame()

    frames = []
    for resource, ticker in RESOURCE_TICKERS.items():
        try:
            s = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True, threads=False)
            if s is None or s.empty:
                continue

            # yfinance may return either simple columns or a MultiIndex such as ('Close', 'CL=F').
            if isinstance(s.columns, pd.MultiIndex):
                close_cols = [col for col in s.columns if 'Close' in col]
                if not close_cols:
                    continue
                close = s[close_cols[0]]
            else:
                if 'Close' not in s.columns:
                    continue
                close = s['Close']

            if isinstance(close, pd.DataFrame):
                close = close.iloc[:, 0]
            tmp = close.rename('close').reset_index()
            date_col = 'Date' if 'Date' in tmp.columns else tmp.columns[0]
            tmp = tmp.rename(columns={date_col: 'date'})
            tmp['resource'] = resource
            tmp = tmp[['date', 'resource', 'close']]
            frames.append(tmp)
        except Exception as e:
            print(f'Could not download {resource} ({ticker}): {e}')
    if not frames:
        return pd.DataFrame()
    out = pd.concat(frames, ignore_index=True)
    out['date'] = pd.to_datetime(out['date']).dt.tz_localize(None)
    out['close'] = pd.to_numeric(out['close'], errors='coerce')
    return out.dropna()


def try_load_world_bank_pink_sheet() -> pd.DataFrame:
    """Optional: load World Bank monthly commodity data. Sheet format may change over time."""
    try:
        wb = pd.read_excel(WORLD_BANK_PINK_SHEET_MONTHLY_XLSX, sheet_name=0, header=None)
        print('World Bank file loaded. Shape:', wb.shape)
        display(wb.head(12))
        return wb
    except Exception as e:
        print('World Bank Pink Sheet download/read failed. This is optional. Error:', e)
        return pd.DataFrame()


def fetch_gdelt_articles(query='energy supply chain disruption', maxrecords=25) -> pd.DataFrame:
    """Fetch recent article metadata from GDELT DOC API. No API key required."""
    url = 'https://api.gdeltproject.org/api/v2/doc/doc'
    params = {
        'query': query,
        'mode': 'artlist',
        'format': 'json',
        'maxrecords': int(maxrecords),
        'sort': 'datedesc',
    }
    try:
        r = requests.get(url, params=params, timeout=25)
        r.raise_for_status()
        data = r.json()
        articles = data.get('articles', [])
        if not articles:
            return pd.DataFrame()
        return pd.DataFrame(articles)
    except Exception as e:
        print('GDELT fetch failed. This is optional. Error:', e)
        return pd.DataFrame()

# Load market data. If online download fails, use sample data.
market_prices = download_yfinance_market_data(start='2016-01-01')
if market_prices.empty:
    print('Using fallback synthetic price data.')
    market_prices = make_sample_market_data()
else:
    print('Loaded online market prices:', market_prices.shape)
    # Ensure every resource exists. Some futures tickers may be temporarily unavailable.
    missing = sorted(set(RESOURCE_TICKERS) - set(market_prices['resource'].unique()))
    if missing:
        print('Adding fallback sample data for missing resources:', missing)
        fallback_prices = make_sample_market_data()
        market_prices = pd.concat([market_prices, fallback_prices[fallback_prices['resource'].isin(missing)]], ignore_index=True)

market_prices.head()

In [ ]:
# Plot recent prices for a quick sanity check.
price_pivot = market_prices.pivot_table(index='date', columns='resource', values='close').sort_index()
recent = price_pivot.tail(500)
ax = recent.plot(figsize=(12, 6), title='Recent resource prices, raw scale')
ax.set_xlabel('Date')
ax.set_ylabel('Price')
plt.show()

# Normalized index chart is easier to compare across resources.
norm = recent / recent.iloc[0] * 100
ax = norm.plot(figsize=(12, 6), title='Recent resource prices indexed to 100')
ax.set_xlabel('Date')
ax.set_ylabel('Index')
plt.show()

In [ ]:
# =====================================================================
# 5. Convert price history into ML features
# =====================================================================

def build_price_features(market_prices: pd.DataFrame) -> pd.DataFrame:
    pivot = market_prices.pivot_table(index='date', columns='resource', values='close').sort_index()
    returns = pivot.pct_change()
    frames = []
    for resource in pivot.columns:
        df = pd.DataFrame({
            'date': pivot.index,
            'resource': resource,
            'price': pivot[resource].values,
        })
        df['return_7d'] = pivot[resource].pct_change(7).values
        df['return_30d'] = pivot[resource].pct_change(30).values
        df['return_90d'] = pivot[resource].pct_change(90).values
        df['vol_30d'] = returns[resource].rolling(30).std().values * np.sqrt(252)
        df['vol_90d'] = returns[resource].rolling(90).std().values * np.sqrt(252)
        df['price_zscore_180d'] = ((pivot[resource] - pivot[resource].rolling(180).mean()) / pivot[resource].rolling(180).std()).values
        frames.append(df)
    feat = pd.concat(frames, ignore_index=True)
    feat = feat.dropna().reset_index(drop=True)
    return feat

price_features = build_price_features(market_prices)
print(price_features.shape)
price_features.head()

In [ ]:
# =====================================================================
# 6. Create a student-demo historical disruption training set
# =====================================================================
# In production, replace this cell with real labeled historical disruption data:
# event date, event type, coordinates, affected assets, actual supply/price impact.


def sample_event_location(resource: str, rng: np.random.Generator) -> Tuple[float, float]:
    relevant = SUPPLY_HUBS[SUPPLY_HUBS['resources'].str.contains(resource, regex=False)]
    if relevant.empty:
        relevant = SUPPLY_HUBS
    hub = relevant.sample(1, random_state=int(rng.integers(0, 1_000_000))).iloc[0]
    # Sample near the hub with some noise.
    lat = float(np.clip(rng.normal(hub['lat'], 8.0), -80, 80))
    lon = float(((rng.normal(hub['lon'], 12.0) + 180) % 360) - 180)
    return lat, lon


def simulate_historical_disruptions(price_features: pd.DataFrame, n_events=4000) -> pd.DataFrame:
    rng = np.random.default_rng(RANDOM_SEED)
    rows = []
    pf = price_features.reset_index(drop=True)
    resources = list(RESOURCE_TICKERS.keys())
    for _ in range(n_events):
        resource = rng.choice(resources)
        event_type = rng.choice(EVENT_TYPES, p=np.array([0.09,0.10,0.09,0.09,0.08,0.08,0.08,0.07,0.07,0.07,0.10,0.08]))
        severity = float(rng.uniform(1, 10))
        duration_days = int(rng.integers(2, 90))
        lat, lon = sample_event_location(resource, rng)
        hub_feat = nearest_hub_features(lat, lon, resource)
        dist_factor = distance_decay(hub_feat['distance_to_hub_km'])
        market_row = pf[pf['resource'].eq(resource)].sample(1, random_state=int(rng.integers(0, 1_000_000))).iloc[0]

        base = EVENT_BASE_IMPACT[event_type]
        sens = RESOURCE_SENSITIVITY[resource]
        hub_mult = 0.55 + hub_feat['hub_importance'] / 10
        duration_mult = np.log1p(duration_days) / np.log1p(90)
        volatility_mult = 0.60 + float(np.nan_to_num(market_row['vol_30d'], nan=0.25))

        # Gold/silver can rise during geopolitical stress even if not physically near the event.
        safe_haven_boost = 0.0
        if resource in {'Gold', 'Silver'} and event_type in SAFE_HAVEN_EVENTS:
            safe_haven_boost = rng.uniform(0.5, 2.5)

        expected_price_pct = (
            0.75 * base * sens * severity * dist_factor * hub_mult * duration_mult * volatility_mult
            + safe_haven_boost
        )
        # Noise and occasional opposite moves reflect market complexity.
        noise = rng.normal(0, 2.5)
        price_delta_30d_pct = float(np.clip(expected_price_pct + noise, -18, 35))

        supply_risk_index = float(np.clip(
            8 * base * sens * (severity / 10) * dist_factor * hub_mult + 0.06 * duration_days + rng.normal(0, 0.5),
            0, 10
        ))

        rows.append({
            'date': market_row['date'],
            'resource': resource,
            'event_type': event_type,
            'severity': severity,
            'duration_days': duration_days,
            'lat': lat,
            'lon': lon,
            'nearest_hub': hub_feat['nearest_hub'],
            'distance_to_hub_km': hub_feat['distance_to_hub_km'],
            'hub_importance': hub_feat['hub_importance'],
            'return_7d': market_row['return_7d'],
            'return_30d': market_row['return_30d'],
            'return_90d': market_row['return_90d'],
            'vol_30d': market_row['vol_30d'],
            'vol_90d': market_row['vol_90d'],
            'price_zscore_180d': market_row['price_zscore_180d'],
            'price_delta_30d_pct': price_delta_30d_pct,
            'supply_risk_index': supply_risk_index,
        })
    return pd.DataFrame(rows)

training_events = simulate_historical_disruptions(price_features, n_events=4500)
print(training_events.shape)
training_events.head()

In [ ]:
# =====================================================================
# 7. Machine Learning forecasting model
# =====================================================================
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.multioutput import MultiOutputRegressor

TARGETS = ['price_delta_30d_pct', 'supply_risk_index']
CATEGORICAL = ['resource', 'event_type', 'nearest_hub']
NUMERIC = [
    'severity', 'duration_days', 'lat', 'lon', 'distance_to_hub_km', 'hub_importance',
    'return_7d', 'return_30d', 'return_90d', 'vol_30d', 'vol_90d', 'price_zscore_180d'
]

X = training_events[CATEGORICAL + NUMERIC]
y = training_events[TARGETS]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.22, random_state=RANDOM_SEED)

preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL),
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), NUMERIC),
])

# Random forest works well for tabular nonlinear shock simulation and is easy to explain to students.
ml_model = Pipeline([
    ('preprocess', preprocess),
    ('model', RandomForestRegressor(n_estimators=250, random_state=RANDOM_SEED, min_samples_leaf=4, n_jobs=-1))
])

ml_model.fit(X_train, y_train)
pred = pd.DataFrame(ml_model.predict(X_test), columns=TARGETS, index=y_test.index)

for target in TARGETS:
    print(f'{target}: MAE={mean_absolute_error(y_test[target], pred[target]):.3f}, R2={r2_score(y_test[target], pred[target]):.3f}')

# Save model for a future app/backend.
import joblib
joblib.dump(ml_model, 'energy_shock_ml_model.joblib')
print('Saved model: energy_shock_ml_model.joblib')

In [ ]:
# Feature importance approximation from the random forest.
# This is useful for students to learn which inputs matter.
try:
    feature_names = ml_model.named_steps['preprocess'].get_feature_names_out()
    importances = ml_model.named_steps['model'].feature_importances_
    importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False)
    display(importance_df.head(20))
except Exception as e:
    print('Feature importance unavailable:', e)

In [ ]:
# =====================================================================
# 8. User-defined scenario forecasting
# =====================================================================
# Change or add events here. Severity uses 1 = minor and 10 = extreme.

USER_EVENTS = [
    {
        'name': 'Major earthquake near Strait of Hormuz',
        'event_type': 'earthquake',
        'lat': 26.6,
        'lon': 56.3,
        'severity': 8.0,
        'duration_days': 21,
        'description': 'A major earthquake damages port and pipeline infrastructure near a critical oil and LNG chokepoint.'
    },
    {
        'name': 'Cyberattack on Gulf Coast refinery network',
        'event_type': 'cyberattack',
        'lat': 29.8,
        'lon': -95.4,
        'severity': 6.5,
        'duration_days': 10,
        'description': 'A coordinated cyberattack disrupts refinery scheduling and logistics systems along the US Gulf Coast.'
    },
]


def latest_market_features(price_features: pd.DataFrame, resource: str) -> pd.Series:
    sub = price_features[price_features['resource'].eq(resource)].sort_values('date')
    if sub.empty:
        raise ValueError(f'No price features for {resource}')
    return sub.iloc[-1]


def scenario_to_model_rows(events: List[dict], resources: Iterable[str]) -> pd.DataFrame:
    rows = []
    for event in events:
        for resource in resources:
            latest = latest_market_features(price_features, resource)
            hub_feat = nearest_hub_features(float(event['lat']), float(event['lon']), resource)
            rows.append({
                'scenario_event': event.get('name', 'Unnamed event'),
                'resource': resource,
                'event_type': event['event_type'],
                'severity': float(event['severity']),
                'duration_days': int(event['duration_days']),
                'lat': float(event['lat']),
                'lon': float(event['lon']),
                'nearest_hub': hub_feat['nearest_hub'],
                'distance_to_hub_km': hub_feat['distance_to_hub_km'],
                'hub_importance': hub_feat['hub_importance'],
                'return_7d': latest['return_7d'],
                'return_30d': latest['return_30d'],
                'return_90d': latest['return_90d'],
                'vol_30d': latest['vol_30d'],
                'vol_90d': latest['vol_90d'],
                'price_zscore_180d': latest['price_zscore_180d'],
            })
    return pd.DataFrame(rows)


def forecast_scenario(events: List[dict], resources: Iterable[str] = None) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if resources is None:
        resources = list(RESOURCE_TICKERS.keys())
    rows = scenario_to_model_rows(events, resources)
    predictions = pd.DataFrame(ml_model.predict(rows[CATEGORICAL + NUMERIC]), columns=TARGETS)
    out = pd.concat([rows.reset_index(drop=True), predictions], axis=1)
    # Combine multiple events. Price effects add only partially; risk uses high-water/weighted behavior.
    agg = out.groupby('resource').agg(
        predicted_30d_price_change_pct=('price_delta_30d_pct', 'sum'),
        max_supply_risk_index=('supply_risk_index', 'max'),
        avg_supply_risk_index=('supply_risk_index', 'mean'),
        nearest_hubs=('nearest_hub', lambda x: ', '.join(sorted(set(x))))
    ).reset_index()
    # Cap additive price forecast for classroom stability.
    agg['predicted_30d_price_change_pct'] = agg['predicted_30d_price_change_pct'].clip(-30, 45)
    agg['risk_level'] = pd.cut(
        agg['max_supply_risk_index'],
        bins=[-0.1, 3.0, 6.5, 10.1],
        labels=['Low', 'Medium', 'High']
    )
    return out, agg.sort_values(['risk_level', 'predicted_30d_price_change_pct'], ascending=[False, False])

scenario_detail, scenario_summary = forecast_scenario(USER_EVENTS)

display(scenario_detail.head(12))
display(scenario_summary)

In [ ]:
# Plot scenario summary.
plot_df = scenario_summary.sort_values('predicted_30d_price_change_pct')
ax = plot_df.plot.barh(x='resource', y='predicted_30d_price_change_pct', figsize=(10, 5), legend=False)
ax.set_title('Scenario forecast: predicted 30-day price change')
ax.set_xlabel('Predicted price change (%)')
ax.set_ylabel('Resource')
plt.show()

risk_df = scenario_summary.sort_values('max_supply_risk_index')
ax = risk_df.plot.barh(x='resource', y='max_supply_risk_index', figsize=(10, 5), legend=False)
ax.set_title('Scenario forecast: maximum supply risk index')
ax.set_xlabel('Supply risk index, 0-10')
ax.set_ylabel('Resource')
plt.show()

In [ ]:
# =====================================================================
# 9. Global map visualization with Folium
# =====================================================================
import folium
from folium.plugins import MarkerCluster


def build_scenario_map(events: List[dict], summary: pd.DataFrame = None):
    m = folium.Map(location=[20, 0], zoom_start=2, tiles='CartoDB positron')
    hub_cluster = MarkerCluster(name='Supply hubs').add_to(m)
    for _, row in SUPPLY_HUBS.iterrows():
        popup = f"<b>{row['hub']}</b><br>Resources: {row['resources']}<br>Importance: {row['importance']}/10"
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=4 + row['importance'] / 2,
            popup=popup,
            fill=True,
            tooltip=row['hub'],
        ).add_to(hub_cluster)

    for event in events:
        popup = (
            f"<b>{event.get('name', 'Event')}</b><br>"
            f"Type: {event['event_type']}<br>Severity: {event['severity']}/10<br>"
            f"Duration: {event['duration_days']} days<br>{event.get('description', '')}"
        )
        folium.Marker(
            location=[event['lat'], event['lon']],
            popup=popup,
            tooltip=event.get('name', event['event_type']),
            icon=folium.Icon(icon='warning-sign', prefix='glyphicon')
        ).add_to(m)
        # Approximate disruption radius: severity * 150 km, converted very roughly to degrees.
        folium.Circle(
            location=[event['lat'], event['lon']],
            radius=float(event['severity']) * 150_000,
            fill=True,
            fill_opacity=0.12,
            popup='Approximate disruption influence zone'
        ).add_to(m)

    folium.LayerControl().add_to(m)
    return m

scenario_map = build_scenario_map(USER_EVENTS, scenario_summary)
scenario_map

In [ ]:
# =====================================================================
# 10. Deep learning / neural-network crisis classification and clustering
# =====================================================================
# This cell creates a demo corpus. Replace with historical event narratives from GDELT,
# ReliefWeb, EIA incident reports, company reports, or curated classroom data.

from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline

EVENT_TEMPLATES = {
    'war': [
        'armed conflict escalates near energy corridor and export terminal',
        'missile strike threatens shipping routes and crude oil exports',
        'regional fighting disrupts refinery operations and border crossings',
    ],
    'earthquake': [
        'major earthquake damages port cranes pipelines roads and power systems',
        'aftershock interrupts mining operations and rail transport',
        'seismic damage forces temporary shutdown of LNG terminal',
    ],
    'hurricane': [
        'hurricane landfall floods refinery district and offshore platforms',
        'storm surge closes ports and delays tanker loading',
        'extreme winds damage power supply to energy infrastructure',
    ],
    'port_closure': [
        'port authority closes harbor after accident and cargo backlog grows',
        'container terminal congestion delays fuel imports and mining equipment',
        'customs disruption slows commodity shipments at critical port',
    ],
    'pipeline_failure': [
        'pipeline leak triggers shutdown and emergency repair crews mobilize',
        'compressor station failure reduces gas flow for several days',
        'pipeline explosion interrupts fuel deliveries to regional markets',
    ],
    'cyberattack': [
        'ransomware attack disrupts refinery scheduling logistics and billing systems',
        'cyber intrusion forces pipeline operator to isolate control network',
        'malware campaign delays port management and commodity documentation',
    ],
    'labor_strike': [
        'workers strike at mine and production output falls sharply',
        'dockworker labor action blocks loading of fuel and metal cargoes',
        'refinery union walkout reduces throughput and shipment reliability',
    ],
    'sanctions': [
        'new sanctions restrict exports insurance and financing for energy cargoes',
        'trade restrictions reduce available supply from major producer',
        'embargo creates uncertainty for buyers shippers and commodity markets',
    ],
    'mine_accident': [
        'mine accident halts extraction and safety investigation begins',
        'tailings incident disrupts copper and gold production schedule',
        'underground collapse forces closure of mineral operation',
    ],
    'drought': [
        'severe drought lowers river levels and limits barge transport capacity',
        'water shortage reduces hydroelectric output and mining processing',
        'dry conditions constrain cooling water for power and refinery assets',
    ],
    'shipping_chokepoint': [
        'vessel incident blocks shipping chokepoint and tanker queue lengthens',
        'naval tensions raise risk for cargo moving through narrow strait',
        'canal closure diverts commodity vessels to longer routes',
    ],
    'pandemic': [
        'public health emergency reduces staffing across ports mines and refineries',
        'pandemic restrictions slow cross-border logistics and inspections',
        'disease outbreak causes demand shock and supply chain uncertainty',
    ],
}


def build_event_text_corpus(n_per_type=80) -> pd.DataFrame:
    rng = np.random.default_rng(RANDOM_SEED)
    regions = ['Middle East', 'Gulf Coast', 'North Sea', 'South America', 'East Asia', 'West Africa', 'Europe', 'Australia']
    resources = list(RESOURCE_TICKERS.keys())
    rows = []
    for label, templates in EVENT_TEMPLATES.items():
        for _ in range(n_per_type):
            text = rng.choice(templates)
            region = rng.choice(regions)
            resource = rng.choice(resources)
            severity = rng.integers(1, 11)
            rows.append({
                'text': f'{text} in {region}; possible impact on {resource}; severity {severity}',
                'event_type': label,
                'severity': severity,
                'resource': resource,
            })
    return pd.DataFrame(rows).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

text_events = build_event_text_corpus()
display(text_events.head())

# Unsupervised clustering with TF-IDF + KMeans.
tfidf = TfidfVectorizer(max_features=1200, ngram_range=(1, 2), stop_words='english')
X_text = tfidf.fit_transform(text_events['text'])
cluster_model = KMeans(n_clusters=6, random_state=RANDOM_SEED, n_init='auto')
text_events['cluster'] = cluster_model.fit_predict(X_text)

# Show top terms per cluster.
terms = np.array(tfidf.get_feature_names_out())
cluster_terms = []
for i, center in enumerate(cluster_model.cluster_centers_):
    top = terms[np.argsort(center)[-10:]][::-1]
    cluster_terms.append({'cluster': i, 'top_terms': ', '.join(top)})
display(pd.DataFrame(cluster_terms))

# Lightweight neural network classifier using sklearn MLP.
X_train_text, X_test_text, y_train_text, y_test_text = train_test_split(
    text_events['text'], text_events['event_type'], test_size=0.25, random_state=RANDOM_SEED, stratify=text_events['event_type']
)

nn_text_classifier = make_pipeline(
    TfidfVectorizer(max_features=1500, ngram_range=(1, 2), stop_words='english'),
    MLPClassifier(hidden_layer_sizes=(96, 48), activation='relu', max_iter=400, random_state=RANDOM_SEED)
)

nn_text_classifier.fit(X_train_text, y_train_text)
y_pred_text = nn_text_classifier.predict(X_test_text)
print('Neural-network text classifier accuracy:', accuracy_score(y_test_text, y_pred_text))
print(classification_report(y_test_text, y_pred_text))

joblib.dump(nn_text_classifier, 'event_text_neural_classifier.joblib')
print('Saved model: event_text_neural_classifier.joblib')

In [ ]:
# Optional TensorFlow/Keras deep learning version.
# Colab usually includes TensorFlow. If it is unavailable, the sklearn neural net above is enough for the demo.
USE_KERAS = True
try:
    import tensorflow as tf
    from tensorflow.keras import layers
except Exception as e:
    USE_KERAS = False
    print('TensorFlow/Keras unavailable; skipping Keras model. Error:', e)

if USE_KERAS:
    labels, label_names = pd.factorize(text_events['event_type'])
    X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
        text_events['text'].values, labels, test_size=0.25, random_state=RANDOM_SEED, stratify=labels
    )
    vectorizer = layers.TextVectorization(max_tokens=2000, output_sequence_length=50)
    vectorizer.adapt(X_train_raw)
    keras_model = tf.keras.Sequential([
        vectorizer,
        layers.Embedding(input_dim=2000, output_dim=32),
        layers.GlobalAveragePooling1D(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.20),
        layers.Dense(len(label_names), activation='softmax')
    ])
    keras_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    history = keras_model.fit(X_train_raw, y_train_raw, validation_split=0.2, epochs=6, batch_size=32, verbose=0)
    loss, acc = keras_model.evaluate(X_test_raw, y_test_raw, verbose=0)
    print('Keras DNN test accuracy:', round(float(acc), 4))
    keras_model.save('keras_event_classifier.keras')
    print('Saved model: keras_event_classifier.keras')

In [ ]:
# Try the classifier on new user text.
new_event_texts = [
    'A ransomware incident forced a fuel pipeline operator to shut down dispatch systems.',
    'A strong earthquake damaged a coastal LNG export terminal and nearby roads.',
    'New export restrictions and insurance limits reduce shipments from a major producer.',
]
for t in new_event_texts:
    print('\nText:', t)
    print('Predicted event type:', nn_text_classifier.predict([t])[0])

In [ ]:
# =====================================================================
# 11. Reinforcement learning response simulator
# =====================================================================
# The RL agent learns a policy for response actions under crisis conditions.
# This is a simplified classroom model, not a real supply-chain optimizer.

ACTIONS = ['do_nothing', 'increase_inventory', 'diversify_suppliers', 'switch_resource', 'demand_response']
STOCK_STATES = ['low', 'medium', 'high']
TREND_STATES = ['falling', 'stable', 'rising']
SEVERITY_STATES = ['low', 'medium', 'high']


def discretize_stock(x):
    return 'low' if x < 0.35 else ('medium' if x < 0.70 else 'high')

def discretize_trend(x):
    return 'falling' if x < -0.02 else ('rising' if x > 0.02 else 'stable')

def discretize_severity(x):
    return 'low' if x < 3.5 else ('medium' if x < 7.0 else 'high')

all_states = [(s, t, v) for s in STOCK_STATES for t in TREND_STATES for v in SEVERITY_STATES]
state_to_idx = {s: i for i, s in enumerate(all_states)}


def simulate_transition(state, action, rng):
    stock, trend, sev = state
    stock_val = {'low': 0.25, 'medium': 0.55, 'high': 0.85}[stock]
    trend_val = {'falling': -0.03, 'stable': 0.0, 'rising': 0.04}[trend]
    sev_val = {'low': 2.0, 'medium': 5.5, 'high': 8.5}[sev]

    action_cost = {
        'do_nothing': 0.0,
        'increase_inventory': 1.8,
        'diversify_suppliers': 1.4,
        'switch_resource': 1.2,
        'demand_response': 0.8,
    }[action]

    # Action effects.
    if action == 'increase_inventory':
        stock_val += 0.20
    elif action == 'diversify_suppliers':
        sev_val -= 1.4
    elif action == 'switch_resource':
        sev_val -= 1.0
        trend_val -= 0.01
    elif action == 'demand_response':
        stock_val += 0.08
        sev_val -= 0.7

    # Random disturbance evolution.
    sev_val += rng.normal(0, 0.8)
    trend_val += rng.normal(0, 0.025)
    stock_val += rng.normal(-0.04, 0.08) - max(sev_val - 5, 0) * 0.025
    stock_val = float(np.clip(stock_val, 0, 1))
    sev_val = float(np.clip(sev_val, 0, 10))

    shortage_penalty = 8.0 * max(0.40 - stock_val, 0)
    price_penalty = 40.0 * max(trend_val, 0) * (sev_val / 10)
    resilience_bonus = 1.5 if (stock_val > 0.55 and sev_val < 6.5) else 0.0
    reward = resilience_bonus - shortage_penalty - price_penalty - action_cost

    next_state = (discretize_stock(stock_val), discretize_trend(trend_val), discretize_severity(sev_val))
    return next_state, reward


def train_q_learning(episodes=5000, alpha=0.15, gamma=0.90, epsilon=0.25):
    rng = np.random.default_rng(RANDOM_SEED)
    Q = np.zeros((len(all_states), len(ACTIONS)))
    for ep in range(episodes):
        state = all_states[int(rng.integers(0, len(all_states)))]
        for _ in range(20):
            s_idx = state_to_idx[state]
            if rng.random() < epsilon:
                a_idx = int(rng.integers(0, len(ACTIONS)))
            else:
                a_idx = int(np.argmax(Q[s_idx]))
            action = ACTIONS[a_idx]
            next_state, reward = simulate_transition(state, action, rng)
            ns_idx = state_to_idx[next_state]
            Q[s_idx, a_idx] += alpha * (reward + gamma * np.max(Q[ns_idx]) - Q[s_idx, a_idx])
            state = next_state
        epsilon = max(0.03, epsilon * 0.999)
    return Q

Q = train_q_learning()
policy = pd.DataFrame([
    {'stock': s[0], 'price_trend': s[1], 'severity': s[2], 'recommended_action': ACTIONS[int(np.argmax(Q[state_to_idx[s]]))]}
    for s in all_states
])
display(policy.head(12))


def recommend_rl_action(resource_summary_row: pd.Series) -> str:
    # Estimate state from forecast summary.
    severity_state = discretize_severity(float(resource_summary_row['max_supply_risk_index']))
    trend_state = discretize_trend(float(resource_summary_row['predicted_30d_price_change_pct']) / 100)
    # Demo assumption: medium stocks unless risk is high.
    stock_state = 'low' if resource_summary_row['max_supply_risk_index'] > 7 else 'medium'
    state = (stock_state, trend_state, severity_state)
    return ACTIONS[int(np.argmax(Q[state_to_idx[state]]))]

scenario_summary['rl_recommended_action'] = scenario_summary.apply(recommend_rl_action, axis=1)
display(scenario_summary[['resource', 'predicted_30d_price_change_pct', 'max_supply_risk_index', 'risk_level', 'rl_recommended_action']])

In [ ]:
# =====================================================================
# 12. Colab mini-app control panel
# =====================================================================
# This is a lightweight notebook UI. For a real website, see the Streamlit cell below.

try:
    import ipywidgets as widgets
    from IPython.display import clear_output

    event_type_w = widgets.Dropdown(options=EVENT_TYPES, value='earthquake', description='Event')
    name_w = widgets.Text(value='New scenario event', description='Name')
    lat_w = widgets.FloatSlider(value=26.6, min=-80, max=80, step=0.1, description='Lat')
    lon_w = widgets.FloatSlider(value=56.3, min=-180, max=180, step=0.1, description='Lon')
    severity_w = widgets.FloatSlider(value=7.0, min=1, max=10, step=0.5, description='Severity')
    duration_w = widgets.IntSlider(value=14, min=1, max=120, step=1, description='Days')
    desc_w = widgets.Textarea(value='Describe the event and possible supply-chain disruption.', description='Text')
    button = widgets.Button(description='Run forecast', button_style='primary')
    output_box = widgets.Output()

    def on_click(_):
        with output_box:
            clear_output(wait=True)
            ev = [{
                'name': name_w.value,
                'event_type': event_type_w.value,
                'lat': lat_w.value,
                'lon': lon_w.value,
                'severity': severity_w.value,
                'duration_days': duration_w.value,
                'description': desc_w.value,
            }]
            detail, summary = forecast_scenario(ev)
            summary['rl_recommended_action'] = summary.apply(recommend_rl_action, axis=1)
            display(summary)
            display(build_scenario_map(ev, summary))

    button.on_click(on_click)
    display(widgets.VBox([name_w, event_type_w, lat_w, lon_w, severity_w, duration_w, desc_w, button, output_box]))
except Exception as e:
    print('Widgets unavailable:', e)

## Development architecture for a real app

A production version should separate the project into layers:

1. **Data layer:** commodity prices, supply nodes, ports, pipelines, mines, weather, news/event feeds, trusted historical labels.
2. **Model layer:** ML forecast model, event classifier, clustering model, RL policy, uncertainty estimates, model monitoring.
3. **API/backend:** FastAPI or Flask service that receives events and returns forecasts.
4. **Frontend web app:** Streamlit for student prototype; React/Next.js for production.
5. **Mobile:** begin as a mobile-friendly web app/PWA, then wrap or rebuild with Swift/React Native/Flutter for App Store submission.
6. **Governance:** citations, data licenses, privacy, safety disclaimers, human review, model cards, and audit logs.

In [ ]:
%%writefile requirements.txt
numpy
pandas
scikit-learn
matplotlib
folium
streamlit
streamlit-folium
yfinance
requests
joblib

In [ ]:
%%writefile README.md
# Energy Resource Shock Simulator

Student-friendly AI prototype for simulating how crisis events may affect energy and commodity supply chains.

## Features
- Global supply-chain event map
- Machine-learning forecast of 30-day price and supply-risk effects
- Event clustering and neural-network event classification
- Reinforcement-learning response recommendations
- Colab notebook and Streamlit starter app

## Run in Colab
1. Upload `Energy_Resource_Shock_AI_App_Colab.ipynb` to Google Colab.
2. Run each cell from top to bottom.
3. Edit `USER_EVENTS` or use the mini-app controls.

## Run as Streamlit app
```bash
pip install -r requirements.txt
streamlit run app.py
```

## Educational disclaimer
This is a classroom simulation. It is not investment advice, emergency guidance, or a validated operational forecast system.

## Suggested citation / credit line
Copyright (c) 2026 [Student Name]. All rights reserved unless a LICENSE file states otherwise.

In [ ]:
%%writefile app.py
import math
import numpy as np
import pandas as pd
import streamlit as st
import folium
from streamlit_folium import st_folium

st.set_page_config(page_title='Energy Resource Shock Simulator', layout='wide')
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

RESOURCE_TICKERS = {
    'WTI Crude Oil': 'CL=F',
    'Brent Crude Oil': 'BZ=F',
    'Natural Gas': 'NG=F',
    'Gold': 'GC=F',
    'Silver': 'SI=F',
    'Copper': 'HG=F',
    'Gasoline': 'RB=F',
    'Heating Oil': 'HO=F',
}
EVENT_TYPES = ['war','earthquake','hurricane','port_closure','pipeline_failure','cyberattack','labor_strike','sanctions','mine_accident','drought','shipping_chokepoint','pandemic']
SUPPLY_HUBS = pd.DataFrame([
    {'hub': 'Strait of Hormuz', 'lat': 26.57, 'lon': 56.25, 'resources': 'WTI Crude Oil,Brent Crude Oil,Natural Gas,Gasoline,Heating Oil', 'importance': 10},
    {'hub': 'Suez Canal', 'lat': 30.59, 'lon': 32.27, 'resources': 'WTI Crude Oil,Brent Crude Oil,Natural Gas,Gasoline,Heating Oil', 'importance': 8},
    {'hub': 'Panama Canal', 'lat': 9.08, 'lon': -79.68, 'resources': 'WTI Crude Oil,Brent Crude Oil,Natural Gas,Copper,Gold', 'importance': 7},
    {'hub': 'US Gulf Coast', 'lat': 29.76, 'lon': -95.37, 'resources': 'WTI Crude Oil,Natural Gas,Gasoline,Heating Oil', 'importance': 9},
    {'hub': 'North Sea', 'lat': 57.0, 'lon': 2.5, 'resources': 'Brent Crude Oil,Natural Gas', 'importance': 8},
    {'hub': 'Chile Copper Belt', 'lat': -22.5, 'lon': -68.9, 'resources': 'Copper,Gold,Silver', 'importance': 9},
    {'hub': 'South Africa Gold Belt', 'lat': -26.2, 'lon': 28.0, 'resources': 'Gold', 'importance': 7},
])
EVENT_BASE_IMPACT = {'war':1.0,'earthquake':0.65,'hurricane':0.70,'port_closure':0.75,'pipeline_failure':0.85,'cyberattack':0.55,'labor_strike':0.45,'sanctions':0.90,'mine_accident':0.60,'drought':0.40,'shipping_chokepoint':0.95,'pandemic':0.60}
RESOURCE_SENSITIVITY = {'WTI Crude Oil':1.00,'Brent Crude Oil':1.05,'Natural Gas':0.90,'Gold':0.55,'Silver':0.45,'Copper':0.70,'Gasoline':0.85,'Heating Oil':0.85}


def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dlambda/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))


def nearest_hub_features(lat, lon, resource):
    subset = SUPPLY_HUBS[SUPPLY_HUBS['resources'].str.contains(resource, regex=False)]
    if subset.empty:
        subset = SUPPLY_HUBS
    distances = haversine_km(lat, lon, subset['lat'].values, subset['lon'].values)
    i = int(np.argmin(distances))
    row = subset.iloc[i]
    return row['hub'], float(distances[i]), float(row['importance'])


def make_training_data(n=3000):
    rng = np.random.default_rng(RANDOM_SEED)
    rows = []
    for _ in range(n):
        resource = rng.choice(list(RESOURCE_TICKERS))
        event_type = rng.choice(EVENT_TYPES)
        severity = rng.uniform(1, 10)
        duration = rng.integers(2, 90)
        hub = SUPPLY_HUBS.sample(1, random_state=int(rng.integers(0, 1_000_000))).iloc[0]
        lat = np.clip(rng.normal(hub['lat'], 8), -80, 80)
        lon = ((rng.normal(hub['lon'], 12) + 180) % 360) - 180
        nearest, distance, importance = nearest_hub_features(lat, lon, resource)
        dist_factor = np.exp(-distance / 1500)
        vol_30d = rng.uniform(0.10, 0.70)
        price_delta = 0.75 * EVENT_BASE_IMPACT[event_type] * RESOURCE_SENSITIVITY[resource] * severity * dist_factor * (0.55 + importance/10) * (np.log1p(duration)/np.log1p(90)) * (0.60 + vol_30d) + rng.normal(0, 2.5)
        risk = np.clip(8 * EVENT_BASE_IMPACT[event_type] * RESOURCE_SENSITIVITY[resource] * (severity/10) * dist_factor * (0.55 + importance/10) + 0.06 * duration + rng.normal(0, 0.5), 0, 10)
        rows.append({'resource':resource,'event_type':event_type,'severity':severity,'duration_days':duration,'lat':lat,'lon':lon,'nearest_hub':nearest,'distance_to_hub_km':distance,'hub_importance':importance,'return_7d':rng.normal(0,0.03),'return_30d':rng.normal(0,0.08),'return_90d':rng.normal(0,0.15),'vol_30d':vol_30d,'vol_90d':rng.uniform(0.10,0.70),'price_zscore_180d':rng.normal(0,1),'price_delta_30d_pct':np.clip(price_delta,-18,35),'supply_risk_index':risk})
    return pd.DataFrame(rows)

@st.cache_resource
def train_model():
    data = make_training_data()
    cat = ['resource','event_type','nearest_hub']
    num = ['severity','duration_days','lat','lon','distance_to_hub_km','hub_importance','return_7d','return_30d','return_90d','vol_30d','vol_90d','price_zscore_180d']
    X = data[cat+num]
    y = data[['price_delta_30d_pct','supply_risk_index']]
    pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat), ('num', Pipeline([('imp', SimpleImputer(strategy='median')),('sc',StandardScaler())]), num)])
    model = Pipeline([('pre', pre), ('rf', RandomForestRegressor(n_estimators=180, min_samples_leaf=4, random_state=RANDOM_SEED, n_jobs=-1))])
    model.fit(X, y)
    return model, cat, num

model, CAT, NUM = train_model()

st.title('Energy Resource Shock Simulator')
st.caption('Student prototype: AI-assisted scenario simulation for commodity supply-chain disruptions.')

with st.sidebar:
    st.header('Scenario event')
    name = st.text_input('Event name', 'New disruption event')
    event_type = st.selectbox('Event type', EVENT_TYPES, index=EVENT_TYPES.index('earthquake'))
    severity = st.slider('Severity', 1.0, 10.0, 7.0, 0.5)
    duration_days = st.slider('Duration, days', 1, 120, 14)
    lat = st.number_input('Latitude', value=26.6, min_value=-80.0, max_value=80.0)
    lon = st.number_input('Longitude', value=56.3, min_value=-180.0, max_value=180.0)
    description = st.text_area('Description', 'Describe the disruption event.')

rows = []
for resource in RESOURCE_TICKERS:
    nearest, dist, importance = nearest_hub_features(lat, lon, resource)
    rows.append({'resource':resource,'event_type':event_type,'severity':severity,'duration_days':duration_days,'lat':lat,'lon':lon,'nearest_hub':nearest,'distance_to_hub_km':dist,'hub_importance':importance,'return_7d':0.0,'return_30d':0.0,'return_90d':0.0,'vol_30d':0.30,'vol_90d':0.35,'price_zscore_180d':0.0})
X = pd.DataFrame(rows)
pred = pd.DataFrame(model.predict(X[CAT+NUM]), columns=['predicted_30d_price_change_pct','supply_risk_index'])
summary = pd.concat([X[['resource','nearest_hub','distance_to_hub_km']], pred], axis=1)
summary['risk_level'] = pd.cut(summary['supply_risk_index'], bins=[-0.1,3,6.5,10.1], labels=['Low','Medium','High'])

col1, col2 = st.columns([1,1])
with col1:
    st.subheader('Forecast summary')
    st.dataframe(summary.sort_values('supply_risk_index', ascending=False), use_container_width=True)
    st.bar_chart(summary.set_index('resource')['predicted_30d_price_change_pct'])
with col2:
    st.subheader('Map')
    m = folium.Map(location=[lat, lon], zoom_start=3, tiles='CartoDB positron')
    folium.Marker([lat, lon], tooltip=name, popup=f'{event_type}, severity {severity}/10').add_to(m)
    folium.Circle([lat, lon], radius=severity*150000, fill=True, fill_opacity=0.12).add_to(m)
    for _, hub in SUPPLY_HUBS.iterrows():
        folium.CircleMarker([hub['lat'], hub['lon']], radius=4+hub['importance']/2, popup=hub['hub'], fill=True).add_to(m)
    st_folium(m, use_container_width=True, height=500)

st.warning('Educational simulation only. Replace synthetic labels with validated historical disruption data before any real use.')

In [ ]:
# Optional: create a zip package for GitHub upload or local development.
import zipfile
from pathlib import Path

package_files = ['README.md', 'requirements.txt', 'app.py', 'energy_shock_ml_model.joblib', 'event_text_neural_classifier.joblib']
zip_path = Path('energy_resource_ai_app_starter.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for filename in package_files:
        if Path(filename).exists():
            zf.write(filename)
print('Created:', zip_path.resolve())

## GitHub, website, iOS, copyright, and promotion checklist

### GitHub launch
1. Create a new GitHub repository.
2. Upload this notebook, `README.md`, `requirements.txt`, and `app.py`.
3. Add screenshots/GIFs of the map and forecast output.
4. Add an issue template: bug report, data-source request, model-improvement request.
5. Add a license only after deciding whether the student wants open source or all-rights-reserved.

### Website app
The fastest free student route is Streamlit Community Cloud:
1. Push `app.py` and `requirements.txt` to GitHub.
2. Create a Streamlit Community Cloud account.
3. Select the GitHub repo, branch, and `app.py` entrypoint.
4. Add secrets only if using API keys such as EIA.

### iOS / Apple App Store route
For a student MVP, do **not** start with a full native app. Start with:
1. Mobile-friendly Streamlit/React web app.
2. PWA-style home-screen install.
3. Later, build a SwiftUI, React Native, Flutter, or Capacitor app that calls the backend API.
4. Apple App Store distribution requires Apple Developer Program enrollment, app review, privacy labels, screenshots, export/privacy compliance checks, and ongoing maintenance.

### Copyright / licensing
- Copyright generally exists automatically when original code/text is fixed in a file.
- Registration can add legal benefits and may be required before bringing a U.S. infringement lawsuit.
- Decide whether the student wants open source (for example, MIT/Apache-2.0) or proprietary/all-rights-reserved.
- Avoid copying proprietary datasets, maps, logos, or model weights without license permission.

### Promotion plan
1. Create a 60-second demo video: user enters earthquake/war event -> map -> forecast -> recommended actions.
2. Publish a GitHub README with screenshots and a clear educational disclaimer.
3. Present at a school research fair or data science club.
4. Write a short blog post explaining ML, clustering, and RL in beginner language.
5. Collect feedback from students and teachers; convert feedback into GitHub issues.

## Next upgrades

- Replace synthetic labels with historical event-impact data.
- Add uncertainty intervals, not only point forecasts.
- Add geospatial layers for ports, pipelines, refineries, mines, power plants, canals, and shipping lanes.
- Add model cards, data cards, and tests.
- Add a backend API so the web/iOS frontends do not train models on every user session.
- Add a moderation/safety layer for crisis scenarios and public outputs.